In [4]:
import polars as pl
import numpy as np
import scipy.stats as stats

In [5]:
ab_data = pl.read_csv("/content/ab_data.csv")
countries = pl.read_csv("/content/countries.csv")

In [6]:
valid_data = ab_data.filter(
    (
        (pl.col("group") == "control") &
        (pl.col("landing_page") == "old_page")
    ) |
    (
        (pl.col("group") == "treatment") &
        (pl.col("landing_page") == "new_page")
    )
)

valid_data = (
    valid_data
    .sort("timestamp")
    .unique(subset=["user_id"], keep="first")
)

df = countries.join(valid_data, on="user_id", how="inner")

In [7]:
print("Sample Sizes & Conversions:")

summary = (
    df.group_by("group")
      .agg(
          pl.len().alias("Users (n)"),
          pl.sum("converted").alias("Conversions")
      )
)

print(summary)

Sample Sizes & Conversions:
shape: (2, 3)
┌───────────┬───────────┬─────────────┐
│ group     ┆ Users (n) ┆ Conversions │
│ ---       ┆ ---       ┆ ---         │
│ str       ┆ u32       ┆ i64         │
╞═══════════╪═══════════╪═════════════╡
│ control   ┆ 57324     ┆ 6869        │
│ treatment ┆ 57565     ┆ 6847        │
└───────────┴───────────┴─────────────┘


In [8]:
control_conv = df.filter(pl.col("group") == "control").select("converted").to_numpy().flatten()
treatment_conv = df.filter(pl.col("group") == "treatment").select("converted").to_numpy().flatten()

mean_c, mean_t = control_conv.mean(), treatment_conv.mean()
diff = mean_t - mean_c

t_res = stats.ttest_ind(treatment_conv, control_conv, equal_var=False)
ci = t_res.confidence_interval(confidence_level=0.95)

n_c, n_t = len(control_conv), len(treatment_conv)
var_c, var_t = control_conv.var(ddof=1), treatment_conv.var(ddof=1)
pooled_std = np.sqrt(((n_t - 1) * var_t + (n_c - 1) * var_c) / (n_t + n_c - 2))
cohens_d = diff / pooled_std

print("T-TEST RESULTS: ")
print(f"Control mean conversion:   {mean_c * 100:.3f}%")
print(f"Treatment mean conversion: {mean_t * 100:.3f}%")
print(f"Difference (T - C):        {diff * 100:.3f} pp")
print(f"95% CI of difference:      [{ci.low * 100:.3f}%, {ci.high * 100:.3f}%]")
print(f"t-statistic:               {t_res.statistic:.3f}")
print(f"p-value:                   {t_res.pvalue:.3f}")
print(f"Cohen's d:                 {cohens_d:.3f}\n")

T-TEST RESULTS: 
Control mean conversion:   11.983%
Treatment mean conversion: 11.894%
Difference (T - C):        -0.088 pp
95% CI of difference:      [-0.463%, 0.287%]
t-statistic:               -0.462
p-value:                   0.644
Cohen's d:                 -0.003



In [9]:
control_data = df.filter(pl.col("group") == "control")
treatment_data = df.filter(pl.col("group") == "treatment")

control_0 = control_data.filter(pl.col("converted") == 0).height
control_1 = control_data.filter(pl.col("converted") == 1).height

treatment_0 = treatment_data.filter(pl.col("converted") == 0).height
treatment_1 = treatment_data.filter(pl.col("converted") == 1).height

contingency_table = np.array([
    [control_0, control_1],
    [treatment_0, treatment_1]
])

chi2, p_val_chi, dof, expected = stats.chi2_contingency(contingency_table)

print("CHI-SQUARE RESULTS:")
print("Contingency Table (Not Converted | Converted):")
print(f"Control:   {control_0} | {control_1}")
print(f"Treatment: {treatment_0} | {treatment_1}")
print(f"chi2-statistic:     {chi2:.3f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value:            {p_val_chi:.3f}\n")

CHI-SQUARE RESULTS:
Contingency Table (Not Converted | Converted):
Control:   50455 | 6869
Treatment: 50718 | 6847
chi2-statistic:     0.205
Degrees of freedom: 1
p-value:            0.651



In [10]:
print("CONVERSION BY COUNTRY:")
country_stats = df.group_by("country").agg([
    pl.len().alias("Users (n)"),
    (pl.col("converted").mean() * 100).round(2).alias("Conversion Rate (%)")
]).sort("country")
print(country_stats)
print()

CONVERSION BY COUNTRY:
shape: (3, 3)
┌─────────┬───────────┬─────────────────────┐
│ country ┆ Users (n) ┆ Conversion Rate (%) │
│ ---     ┆ ---       ┆ ---                 │
│ str     ┆ u32       ┆ f64                 │
╞═════════╪═══════════╪═════════════════════╡
│ CA      ┆ 5718      ┆ 11.79               │
│ UK      ┆ 28739     ┆ 12.32               │
│ US      ┆ 80432     ┆ 11.81               │
└─────────┴───────────┴─────────────────────┘



In [11]:
ca_conv = df.filter(pl.col("country") == "CA").select("converted").to_numpy().flatten()
uk_conv = df.filter(pl.col("country") == "UK").select("converted").to_numpy().flatten()
us_conv = df.filter(pl.col("country") == "US").select("converted").to_numpy().flatten()

f_stat_country, p_val_country = stats.f_oneway(ca_conv, uk_conv, us_conv)

df = df.with_columns((pl.col("country") + "_" + pl.col("group")).alias("country_strategy"))
group_arrays = [
    group_df.select("converted").to_numpy().flatten()
    for name, group_df in df.group_by("country_strategy")
]
f_stat_combo, p_val_combo = stats.f_oneway(*group_arrays)

print("ANOVA RESULTS:")
print(f"Across 3 Countries -> F-stat: {f_stat_country:.3f}, p-value: {p_val_country:.3f}")
print(f"Across 6 Groups    -> F-stat: {f_stat_combo:.3f}, p-value: {p_val_combo:.3f}")

ANOVA RESULTS:
Across 3 Countries -> F-stat: 2.624, p-value: 0.073
Across 6 Groups    -> F-stat: 1.422, p-value: 0.213
